In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

import tensorflow as tf
from tensorflow import keras
from keras import layers, models, optimizers, applications, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.utils import to_categorical
from keras.preprocessing import image

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FOUND: /content/best_model.keras
Not found: /content/drive/MyDrive/best_model.keras


In [3]:
import os

base_path = "/content/drive/MyDrive/real vs fake images"

print(os.listdir(base_path))

['FAKE', 'REAL']


In [11]:
import os
from PIL import Image
from collections import Counter

base_path = "/content/drive/MyDrive/real vs fake images"

for label in ["REAL", "FAKE"]:
    folder = os.path.join(base_path, label)

    files = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]

    print(f"\n{label}: {len(files)} images")

    resolutions = Counter()

    for file in files:
        path = os.path.join(folder, file)

        try:
            with Image.open(path) as img:
                resolutions[img.size] += 1
        except:
            print("Problem with:", file)

    print("Top resolutions:")
    for resolution, count in resolutions.most_common(10):
        print(f"  {resolution}: {count}")


REAL: 11705 images
Top resolutions:
  (32, 32): 10000
  (512, 512): 1705

FAKE: 11721 images
Top resolutions:
  (32, 32): 10000
  (512, 512): 1721


In [4]:
import os
import pandas as pd

data = []

for label in ["REAL", "FAKE"]:
    folder = os.path.join(base_path, label)

    for file in os.listdir(folder):
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            data.append({
                "path": os.path.join(folder, file),
                "label": label
            })

df = pd.DataFrame(data)

print(df["label"].value_counts())

label
FAKE    11721
REAL    11705
Name: count, dtype: int64


In [5]:
df["label"] = df["label"].map({
    "REAL": 0,
    "FAKE": 1
})

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
CLASS_MODE = "binary"

In [8]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

In [9]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["path"].values, val_df["label"].values)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["path"].values, test_df["label"].values)
)

In [10]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [11]:
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Pixel range:", tf.reduce_min(images).numpy(),
          "to", tf.reduce_max(images).numpy())

Images shape: (64, 224, 224, 3)
Labels shape: (64,)
Pixel range: 0.0 to 1.0


In [12]:
def build_classifier_head(inputs, dropout_rate=0.3):

    model = keras.Sequential()

    model.add(layers.Flatten())

    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.4))

    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.3))

    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.2))

    model.add(layers.Dense(1, activation="sigmoid"))

    return model(inputs)

In [13]:
# Custom CNN feature extractor + shared classifier head
def build_simple_cnn(img_shape):

    inputs = layers.Input(shape=img_shape)

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Classifier Head
    out = build_classifier_head(x, dropout_rate=0.3)

    model = models.Model(
        inputs,
        out,
        name="CustomCNN"
    )

    return model

In [14]:
mlp_input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
cnn_model = build_simple_cnn(mlp_input_shape)
cnn_model.compile(optimizer=optimizers.Adam(1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

Model: "CustomCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             

 Total params: 27,034,145 (103.13 MB)

 Trainable params: 27,030,689 (103.11 MB)

 Non-trainable params: 3,456 (13.50 KB)

In [15]:
def build_transfer_model(base_model_name, img_shape=(224, 224, 3)):

    inputs = layers.Input(shape=img_shape)

    # Convert [0,1] back to [0,255]
    x = layers.Rescaling(255.0)(inputs)

    if base_model_name == "VGG16":
        x = applications.vgg16.preprocess_input(x)
        base = applications.VGG16(
            weights="imagenet",
            include_top=False,
            input_tensor=x
        )

    elif base_model_name == "ResNet50":
        x = applications.resnet50.preprocess_input(x)
        base = applications.ResNet50(
            weights="imagenet",
            include_top=False,
            input_tensor=x
        )

    else:
        raise ValueError("Unsupported base model")

    base.trainable = False

    features = base.output
    features = layers.GlobalAveragePooling2D()(features)

    out = build_classifier_head(
        features,
        dropout_rate=0.3
    )

    model = models.Model(
        inputs=inputs,
        outputs=out,
        name=f"{base_model_name}_with_shared_head"
    )

    model.compile(
        optimizer=optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [16]:
vgg16_model = build_transfer_model("VGG16", img_shape=mlp_input_shape)
resnet50_model = build_transfer_model("ResNet50", img_shape=mlp_input_shape)
vgg16_model.summary()
resnet50_model.summary()

Model: "VGG16_with_shared_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ rescaling[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ rescaling[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ rescaling[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 224, 224,  │      1,792 │ add[0][0]         │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 224, 224,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 112, 112,  │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 112, 112,  │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 112, 112,  │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 56, 56,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 56, 56,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 56, 56,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 56, 56,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 28, 28,    │          0 │ block3_conv3[0][

 Total params: 15,144,769 (57.77 MB)

 Trainable params: 428,545 (1.63 MB)

 Non-trainable params: 14,716,224 (56.14 MB)

Model: "ResNet50_with_shared_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ rescaling_1[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ rescaling_1[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ rescaling_1[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ add_1[0][0]       │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 24,804,225 (94.62 MB)

 Trainable params: 1,214,977 (4.63 MB)

 Non-trainable params: 23,589,248 (89.99 MB)

In [17]:
EPOCHS = 20

def train_and_evaluate(model, name):

    checkpoint_path = os.path.join(
        MODEL_DIR,
        f"{name}_best.keras"
    )

    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    )

    early_cb = callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )

    print(f"\nTraining {name}...")
    print(f"Checkpoint: {checkpoint_path}")

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[
            checkpoint_cb,
            early_cb
        ]
    )

    results = model.evaluate(
        test_ds,
        return_dict=True
    )

    print(f"\n{name} test results:")
    print(results)

    return history, results

In [40]:
# train cnn
cnn_history, cnn_results = train_and_evaluate(cnn_model, "CustomCNN")



Training CustomCNN...
Epoch 1/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 200s 534ms/step - accuracy: 0.7802 - loss: 0.4668 - val_accuracy: 0.5003 - val_loss: 1.3360
Epoch 2/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 114s 438ms/step - accuracy: 0.8652 - loss: 0.3152 - val_accuracy: 0.7399 - val_loss: 0.5380
Epoch 3/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 122s 466ms/step - accuracy: 0.8894 - loss: 0.2654 - val_accuracy: 0.8822 - val_loss: 0.2814
Epoch 4/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 118s 451ms/step - accuracy: 0.9085 - loss: 0.2225 - val_accuracy: 0.8944 - val_loss: 0.2771
Epoch 5/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 118s 449ms/step - accuracy: 0.9172 - loss: 0.2046 - val_accuracy: 0.9069 - val_loss: 0.2342
Epoch 6/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 113s 434ms/step - accuracy: 0.9276 - loss: 0.1749 - val_accuracy: 0.8998 - val_loss: 0.2612
Epoch 7/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 120s 462ms/step - accuracy: 0.9417 - loss: 0.1485 - val_accuracy: 0.9138 - val_loss: 0.2257
Epoch 8/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 114s 434ms/st

In [18]:
from tensorflow import keras

cnn_model = keras.models.load_model(
    "/content/drive/MyDrive/real_vs_fake_models/CustomCNN.keras",
    compile=False
)

cnn_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [19]:
cnn_results = cnn_model.evaluate(
    test_ds,
    return_dict=True
)

print(cnn_results)

55/55 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.9377 - loss: 0.1606
{'accuracy': 0.9376778602600098, 'loss': 0.16057425737380981}


In [4]:
import os

MODEL_DIR = "/content/drive/MyDrive/real_vs_fake_models"
os.makedirs(MODEL_DIR, exist_ok=True)

old_model.save(
    os.path.join(MODEL_DIR, "CustomCNN.keras")
)

print("Custom CNN saved to Google Drive successfully.")

Custom CNN saved to Google Drive successfully.


In [ ]:
# train pretrained models
vgg_history, vgg_results = train_and_evaluate(vgg16_model, "VGG16_transfer")
resnet_history, resnet_results = train_and_evaluate(resnet50_model, "ResNet50_transfer")



Training VGG16_transfer...
Checkpoint: /content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras
Epoch 1/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - accuracy: 0.6499 - loss: 0.6850
Epoch 1: val_loss improved from None to 0.31480, saving model to /content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras
257/257 ━━━━━━━━━━━━━━━━━━━━ 153s 547ms/step - accuracy: 0.7345 - loss: 0.5371 - val_accuracy: 0.8677 - val_loss: 0.3148
Epoch 2/20
257/257 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accuracy: 0.8416 - loss: 0.3587
Epoch 2: val_loss improved from 0.31480 to 0.24485, saving model to /content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras
257/257 ━━━━━━━━━━━━━━━━━━━━ 116s 443ms/step - accuracy: 0.8531 - loss: 0.3369 - val_accuracy: 0.8990 - val_loss: 0

In [20]:
vgg16_model = keras.models.load_model(
    "/content/drive/MyDrive/real_vs_fake_models/VGG16_transfer_best.keras",
    compile=False
)

vgg16_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

vgg_results = vgg16_model.evaluate(
    test_ds,
    return_dict=True
)

print(vgg_results)

55/55 ━━━━━━━━━━━━━━━━━━━━ 38s 487ms/step - accuracy: 0.9360 - loss: 0.1646
{'accuracy': 0.9359704256057739, 'loss': 0.16459359228610992}


In [21]:
resnet50_model = keras.models.load_model(
    "/content/drive/MyDrive/real_vs_fake_models/ResNet50_transfer_best.keras",
    compile=False
)

resnet50_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

resnet_results = resnet50_model.evaluate(
    test_ds,
    return_dict=True
)

print(resnet_results)

55/55 ━━━━━━━━━━━━━━━━━━━━ 25s 276ms/step - accuracy: 0.9525 - loss: 0.1279
{'accuracy': 0.9524757862091064, 'loss': 0.12788206338882446}


In [22]:
import numpy as np

def ensemble_predict(images):

    cnn_pred = cnn_model.predict(images, verbose=0).reshape(-1)
    vgg_pred = vgg16_model.predict(images, verbose=0).reshape(-1)
    resnet_pred = resnet50_model.predict(images, verbose=0).reshape(-1)

    # Weighted ensemble
    ensemble_pred = (
        0.20 * cnn_pred +
        0.30 * vgg_pred +
        0.50 * resnet_pred
    )

    return ensemble_pred

In [23]:
y_true = []
y_pred = []

for images, labels in test_ds:

    ensemble_probs = ensemble_predict(images)

    y_true.extend(labels.numpy())
    y_pred.extend(ensemble_probs)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

ensemble_labels = (y_pred >= 0.5).astype(int)

ensemble_accuracy = np.mean(ensemble_labels == y_true)

print("Ensemble Accuracy:", ensemble_accuracy)

Ensemble Accuracy: 0.9655663062037564


In [27]:
image_path = "/0000 (8).jpg"

In [28]:
from PIL import Image
import numpy as np

# Load image
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))

# Preprocessing
image = np.array(image, dtype=np.float32) / 255.0
image = np.expand_dims(image, axis=0)

# Predictions
cnn_pred = cnn_model.predict(image, verbose=0)[0][0]
vgg_pred = vgg16_model.predict(image, verbose=0)[0][0]
resnet_pred = resnet50_model.predict(image, verbose=0)[0][0]

# Ensemble
ensemble_pred = (
    0.20 * cnn_pred +
    0.30 * vgg_pred +
    0.50 * resnet_pred
)

def result(prob):
    label = "FAKE" if prob >= 0.5 else "REAL"
    confidence = prob * 100 if prob >= 0.5 else (1 - prob) * 100
    return label, confidence

print(f"Custom CNN : {result(cnn_pred)[0]} ({result(cnn_pred)[1]:.2f}%)")
print(f"VGG16      : {result(vgg_pred)[0]} ({result(vgg_pred)[1]:.2f}%)")
print(f"ResNet50   : {result(resnet_pred)[0]} ({result(resnet_pred)[1]:.2f}%)")
print(f"ENSEMBLE   : {result(ensemble_pred)[0]} ({result(ensemble_pred)[1]:.2f}%)")

Custom CNN : REAL (98.81%)
VGG16      : FAKE (90.14%)
ResNet50   : REAL (95.94%)
ENSEMBLE   : REAL (70.69%)
